In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style("ticks")
plt.rcParams['figure.dpi'] = 100

## 1. Fetch Data from W&B

Configure your W&B project and run details below:

In [ ]:
# W&B Configuration
WANDB_PROJECT = "exact"
WANDB_ENTITY = "assistive-autonomy"

# Run IDs for executable activity model assessment
# These will be populated after running assessment_exec.py
RUN_ID_VERBS = None      # Fine-grained (verbs) - e.g., "abc123"
RUN_ID_ACTIVITY = None   # Coarse-grained (activity) - e.g., "def456"

# Or search by name pattern
RUN_NAME_PATTERN = "exec_"  # Matches runs like "exec_verbs_20260128_..."

In [ ]:
def fetch_wandb_exec_data(project: str, entity: str, run_id: str = None, 
                          run_name_pattern: str = None, label_type: str = None):
    """
    Fetch executable activity model assessment data from W&B.
    
    Args:
        project: W&B project name
        entity: W&B entity (org/user)
        run_id: Specific run ID to fetch
        run_name_pattern: Pattern to match run name (e.g., "exec_verbs")
        label_type: Filter by label type ("verbs" or "activity")
        
    Returns:
        dict with run data, summary, config, tables
    """
    api = wandb.Api()
    
    # Find the run
    if run_id:
        run = api.run(f"{entity}/{project}/{run_id}")
    else:
        # Search for matching runs
        runs = api.runs(f"{entity}/{project}")
        
        # Filter by pattern and label_type
        matching = []
        for r in runs:
            if run_name_pattern and run_name_pattern not in r.name:
                continue
            if label_type and r.config.get("label_type") != label_type:
                continue
            if r.config.get("method") != "executable_activity_models":
                continue
            matching.append(r)
        
        if not matching:
            raise ValueError(f"No runs found matching criteria")
        
        # Sort by created_at and get most recent
        matching.sort(key=lambda r: r.created_at, reverse=True)
        run = matching[0]
    
    print(f"Found run: {run.name} (ID: {run.id})")
    print(f"State: {run.state}")
    print(f"Label type: {run.config.get('label_type', 'unknown')}")
    
    # Get summary metrics
    summary = run.summary._json_dict
    
    # Fetch tables from artifacts
    results_df = None
    matrix_df = None
    
    try:
        for artifact in run.logged_artifacts():
            if artifact.type == "run_table":
                for file in artifact.files():
                    if "results_table" in file.name:
                        table_path = artifact.download()
                        table_file = Path(table_path) / file.name
                        with open(table_file) as f:
                            table_data = json.load(f)
                        results_df = pd.DataFrame(table_data["data"], columns=table_data["columns"])
                        print(f"Loaded results_table with {len(results_df)} activities")
                    
                    if "separability_matrix" in file.name:
                        table_path = artifact.download()
                        table_file = Path(table_path) / file.name
                        with open(table_file) as f:
                            table_data = json.load(f)
                        matrix_df = pd.DataFrame(table_data["data"], columns=table_data["columns"])
                        print(f"Loaded separability_matrix ({len(matrix_df)}x{len(matrix_df)})")
    except Exception as e:
        print(f"Could not fetch tables from artifacts: {e}")
    
    return {
        "run": run,
        "summary": summary,
        "config": run.config,
        "results_df": results_df,
        "matrix_df": matrix_df,
    }


def parse_exec_data(wandb_data: dict):
    """
    Parse W&B data into format for plotting.
    
    Returns:
        dict with activities, distance_matrix, separation_vector, etc.
    """
    results_df = wandb_data["results_df"]
    matrix_df = wandb_data["matrix_df"]
    summary = wandb_data["summary"]
    
    # Get activities from matrix
    activities = matrix_df["model_trained_on"].tolist()
    n = len(activities)
    
    # Build distance matrix
    distance_matrix = np.zeros((n, n))
    for i, row in matrix_df.iterrows():
        for j, act in enumerate(activities):
            val = row[act]
            distance_matrix[i, j] = val if val is not None else np.nan
    
    # Build separation vector
    separation_vector = np.full(n, np.nan)
    same_dist_vector = np.full(n, np.nan)
    cross_dist_vector = np.full(n, np.nan)
    
    for i, act in enumerate(activities):
        match = results_df[results_df["activity"] == act]
        if len(match) > 0:
            separation_vector[i] = match["separation"].iloc[0] or np.nan
            same_dist_vector[i] = match["same_activity_dist"].iloc[0] or np.nan
            cross_dist_vector[i] = match["cross_activity_dist"].iloc[0] or np.nan
    
    return {
        "activities": activities,
        "distance_matrix": distance_matrix,
        "separation_vector": separation_vector,
        "same_dist_vector": same_dist_vector,
        "cross_dist_vector": cross_dist_vector,
        "label_type": wandb_data["config"].get("label_type", "unknown"),
        "train_budget": wandb_data["config"].get("train_budget", "?"),
    }

In [ ]:
# Alternative: Load from local results file
def load_local_results(results_path: str):
    """
    Load assessment results from local JSON file.
    
    Args:
        results_path: Path to results.json
        
    Returns:
        dict in same format as parse_exec_data output
    """
    with open(results_path, "r") as f:
        data = json.load(f)
    
    activities = data["activity_names"]
    n = len(activities)
    
    # Build matrix
    distance_matrix = np.array(data["matrix"])
    
    # Build vectors
    metrics = data["metrics"]
    per_activity = metrics.get("per_activity", {})
    
    separation_vector = np.array([per_activity.get(a, {}).get("separation", np.nan) for a in activities])
    same_dist_vector = np.array([per_activity.get(a, {}).get("same_activity_dist", np.nan) for a in activities])
    cross_dist_vector = np.array([per_activity.get(a, {}).get("cross_activity_dist", np.nan) for a in activities])
    
    config = data.get("config", {})
    
    return {
        "activities": activities,
        "distance_matrix": distance_matrix,
        "separation_vector": separation_vector,
        "same_dist_vector": same_dist_vector,
        "cross_dist_vector": cross_dist_vector,
        "label_type": config.get("label_type", "unknown"),
        "train_budget": config.get("train_budget", "?"),
        "metrics": metrics,
    }

## 2. Load Assessment Results

Choose to load from W&B or local files:

In [ ]:
# === OPTION A: Load from W&B ===
# Uncomment to fetch from wandb

# data_verbs = fetch_wandb_exec_data(WANDB_PROJECT, WANDB_ENTITY, label_type="verbs")
# data_verbs = parse_exec_data(data_verbs)

# data_activity = fetch_wandb_exec_data(WANDB_PROJECT, WANDB_ENTITY, label_type="activity")
# data_activity = parse_exec_data(data_activity)

In [ ]:
# === OPTION B: Load from local files ===
# Update paths to your results

RESULTS_DIR = Path("../results/assessment_exec")

# Find latest results for each label type
def find_latest_results(base_dir: Path, label_type: str):
    """Find most recent results directory for a label type."""
    matching = sorted(base_dir.glob(f"{label_type}_*"), reverse=True)
    if matching:
        return matching[0] / "results.json"
    return None

# Try to load verbs results
verbs_path = find_latest_results(RESULTS_DIR, "verbs")
if verbs_path and verbs_path.exists():
    data_verbs = load_local_results(str(verbs_path))
    print(f"Loaded verbs data from {verbs_path}")
    print(f"  Activities: {len(data_verbs['activities'])}")
    print(f"  Train budget: {data_verbs['train_budget']}")
else:
    data_verbs = None
    print("No verbs results found - run assessment_exec.py first")

# Try to load activity results
activity_path = find_latest_results(RESULTS_DIR, "activity")
if activity_path and activity_path.exists():
    data_activity = load_local_results(str(activity_path))
    print(f"\nLoaded activity data from {activity_path}")
    print(f"  Activities: {len(data_activity['activities'])}")
    print(f"  Train budget: {data_activity['train_budget']}")
else:
    data_activity = None
    print("\nNo activity results found - run assessment_exec.py first")

## 3. Visualization Functions

In [ ]:
# === PLOT CONFIGURATION ===

# Figure settings
FIGSIZE_MATRIX = (12, 10)
FIGSIZE_SUMMARY = (10, 6)
DPI = 150

# Color palette
CMAP = 'coolwarm'  # Blue=low distance (similar), Red=high distance (different)

# Font settings
TITLE_FONTSIZE = 16
LABEL_FONTSIZE = 14
TICK_FONTSIZE = 11
ANNOT_FONTSIZE = 10

# Bar chart colors
GOOD_COLOR = '#2ecc71'   # Green for positive separation
BAD_COLOR = '#e74c3c'    # Red for negative separation

In [ ]:
def plot_distance_matrix(
    data: dict,
    title: str = None,
    figsize: tuple = FIGSIZE_MATRIX,
    cmap: str = CMAP,
    save_path: str = None,
):
    """
    Plot program edit distance separability matrix.
    
    For edit distance: lower values = more similar programs.
    Diagonal should be LOW (same-activity programs are similar).
    Off-diagonal should be HIGH (cross-activity programs are different).
    """
    activities = data["activities"]
    matrix = data["distance_matrix"]
    label_type = data.get("label_type", "")
    n = len(activities)
    
    # Wrap labels for display
    wrapped = [a.replace("_", "\n").replace(" ", "\n") for a in activities]
    
    if title is None:
        title = f"Program Edit Distance Matrix ({label_type})"
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Heatmap - note: for edit distance, lower is better on diagonal
    im = sns.heatmap(
        matrix,
        annot=True,
        fmt=".1f",
        cmap=cmap,
        square=True,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Min Edit Distance (lower = more similar)", "shrink": 0.8},
        annot_kws={"size": ANNOT_FONTSIZE},
        ax=ax,
        xticklabels=wrapped,
        yticklabels=wrapped,
    )
    
    ax.set_xlabel("Model Activity", fontsize=LABEL_FONTSIZE, fontweight="bold")
    ax.set_ylabel("Query Activity", fontsize=LABEL_FONTSIZE, fontweight="bold")
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold")
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor="white")
        print(f"Saved to: {save_path}")
    
    plt.show()
    return fig

In [ ]:
def plot_separation_bars(
    data: dict,
    title: str = None,
    figsize: tuple = FIGSIZE_SUMMARY,
    save_path: str = None,
):
    """
    Plot per-activity separation scores as bar chart.
    
    Separation = cross_activity_dist - same_activity_dist
    Positive = good (different activities have higher distance)
    """
    activities = data["activities"]
    separation = data["separation_vector"]
    label_type = data.get("label_type", "")
    n = len(activities)
    
    if title is None:
        title = f"Per-Activity Separation ({label_type})"
    
    # Colors based on positive/negative
    colors = [GOOD_COLOR if s > 0 else BAD_COLOR for s in separation]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    bars = ax.bar(
        np.arange(n),
        separation,
        color=colors,
        edgecolor="black",
        linewidth=0.5,
    )
    
    ax.set_xticks(np.arange(n))
    ax.set_xticklabels(activities, rotation=45, ha="right", fontsize=TICK_FONTSIZE)
    ax.set_ylabel("Separation (cross - same distance)", fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.axhline(y=0, color="black", linestyle="-", linewidth=1, alpha=0.7)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    
    # Value labels
    for i, (bar, val) in enumerate(zip(bars, separation)):
        if not np.isnan(val):
            offset = 0.1 if val > 0 else -0.1
            ax.annotate(
                f"{val:.2f}",
                xy=(i, val + offset),
                ha="center",
                va="bottom" if val > 0 else "top",
                fontsize=8,
            )
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor="white")
        print(f"Saved to: {save_path}")
    
    plt.show()
    return fig

In [ ]:
def plot_distance_comparison(
    data: dict,
    title: str = None,
    figsize: tuple = (12, 5),
    save_path: str = None,
):
    """
    Plot same-activity vs cross-activity distances side by side.
    """
    activities = data["activities"]
    same_dist = data["same_dist_vector"]
    cross_dist = data["cross_dist_vector"]
    label_type = data.get("label_type", "")
    n = len(activities)
    
    if title is None:
        title = f"Same vs Cross Activity Distance ({label_type})"
    
    x = np.arange(n)
    width = 0.35
    
    fig, ax = plt.subplots(figsize=figsize)
    
    bars1 = ax.bar(x - width/2, same_dist, width, label="Same Activity", color="#3498db", edgecolor="black")
    bars2 = ax.bar(x + width/2, cross_dist, width, label="Cross Activity", color="#e74c3c", edgecolor="black")
    
    ax.set_xlabel("Activity", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Mean Min Edit Distance", fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(activities, rotation=45, ha="right", fontsize=TICK_FONTSIZE)
    ax.legend(fontsize=12)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor="white")
        print(f"Saved to: {save_path}")
    
    plt.show()
    return fig

## 4. Fine-Grained Analysis (Verbs)

In [ ]:
if data_verbs:
    print("=" * 60)
    print("Fine-Grained Activity Recognition (Verbs)")
    print("=" * 60)
    print(f"Activities: {len(data_verbs['activities'])}")
    print(f"Train budget: {data_verbs['train_budget']} programs/activity")
    
    metrics = data_verbs.get("metrics", {})
    print(f"\nOverall Metrics:")
    print(f"  Diagonal mean (same-activity): {metrics.get('diagonal_mean', 'N/A'):.2f}")
    print(f"  Off-diagonal mean (cross-activity): {metrics.get('off_diagonal_mean', 'N/A'):.2f}")
    print(f"  Separation: {metrics.get('separation', 'N/A'):.2f}")
else:
    print("No verbs data available")

In [ ]:
if data_verbs:
    fig = plot_distance_matrix(data_verbs, title="Program Edit Distance (Fine-Grained Verbs)")

In [ ]:
if data_verbs:
    fig = plot_separation_bars(data_verbs, title="Per-Activity Separation (Verbs)")

In [ ]:
if data_verbs:
    fig = plot_distance_comparison(data_verbs, title="Same vs Cross Activity Distance (Verbs)")

## 5. Coarse-Grained Analysis (Activity)

In [ ]:
if data_activity:
    print("=" * 60)
    print("Coarse-Grained Activity Recognition (Activity)")
    print("=" * 60)
    print(f"Activities: {len(data_activity['activities'])}")
    print(f"  {data_activity['activities']}")
    print(f"Train budget: {data_activity['train_budget']} programs/activity")
    
    metrics = data_activity.get("metrics", {})
    print(f"\nOverall Metrics:")
    print(f"  Diagonal mean (same-activity): {metrics.get('diagonal_mean', 'N/A'):.2f}")
    print(f"  Off-diagonal mean (cross-activity): {metrics.get('off_diagonal_mean', 'N/A'):.2f}")
    print(f"  Separation: {metrics.get('separation', 'N/A'):.2f}")
else:
    print("No activity data available")

In [ ]:
if data_activity:
    fig = plot_distance_matrix(data_activity, title="Program Edit Distance (Coarse-Grained Activity)")

In [ ]:
if data_activity:
    fig = plot_separation_bars(data_activity, title="Per-Activity Separation (Activity)")

In [ ]:
if data_activity:
    fig = plot_distance_comparison(data_activity, title="Same vs Cross Activity Distance (Activity)")

## 6. Comparison: Fine-Grained vs Coarse-Grained

In [ ]:
if data_verbs and data_activity:
    print("=" * 60)
    print("Comparison: Fine-Grained (Verbs) vs Coarse-Grained (Activity)")
    print("=" * 60)
    
    metrics_v = data_verbs.get("metrics", {})
    metrics_a = data_activity.get("metrics", {})
    
    comparison = pd.DataFrame({
        "Metric": ["N Activities", "Diagonal Mean", "Off-Diagonal Mean", "Separation"],
        "Verbs (Fine)": [
            len(data_verbs["activities"]),
            f"{metrics_v.get('diagonal_mean', 0):.2f}",
            f"{metrics_v.get('off_diagonal_mean', 0):.2f}",
            f"{metrics_v.get('separation', 0):.2f}",
        ],
        "Activity (Coarse)": [
            len(data_activity["activities"]),
            f"{metrics_a.get('diagonal_mean', 0):.2f}",
            f"{metrics_a.get('off_diagonal_mean', 0):.2f}",
            f"{metrics_a.get('separation', 0):.2f}",
        ],
    })
    
    display(comparison)

## 7. Interpretation Guide

### How to Read the Distance Matrix

The matrix shows **mean minimum edit distance** values:
- **Rows** = Query activity (test programs from this activity)
- **Columns** = Model activity (programs in this activity's model)
- **Cell value** = Average of min edit distances from each query to all model programs

| Pattern | Interpretation |
|---------|---------------|
| **Low diagonal (blue)** | Same-activity programs are structurally similar ✓ |
| **High off-diagonal (red)** | Cross-activity programs are structurally different ✓ |
| **Uniform row** | Activity has ambiguous programs (bad) |
| **Low column** | One activity's model matches many queries (over-general) |

### Edit Distance Intuition

The Unordered Tree Edit Distance (UTED) measures structural similarity between programs:

- **Lower distance** = More similar program structure
- **Higher distance** = More different program structure

Programs are compared ignoring temporal intervals, focusing on:
- Which joints/sensors are used
- The values (with tolerance of 0.3)
- The tree structure (conjunctions, sequences)

### Separation Metric

$$\text{Separation} = \bar{d}_{\text{cross}} - \bar{d}_{\text{same}}$$

- **Positive (green)**: Cross-activity distance > same-activity distance ✓
- **Negative (red)**: Same-activity programs are MORE different than cross-activity ✗
- **Larger positive** = Better discrimination

### Common Issues

| Issue | Possible Cause |
|-------|---------------|
| Low separation for all | Parser generating too generic programs |
| Negative separation | Activity has high intra-class variability |
| One activity dominates | Over-represented in training data |
| Uniform matrix | Programs lack discriminative structure |